In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy import stats
from nba_api.stats.endpoints import leaguedashteamstats
from datetime import datetime

pd.set_option('display.max_columns', None)

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

In [2]:
from src.points_model import PointsPropModel
from src.points_model.utils import generate_synthetic_game_logs

# Generate test data
game_logs = generate_synthetic_game_logs(n_players=50, games_per_player=30)

# Test your model
model = PointsPropModel()
model.fit(game_logs)

# Test projections
projection = model.project_points(player_id=1, is_b2b=False)
pd.DataFrame(projection)

,player_id,expected_points,std,lower_90,upper_90,components,adjustments,breakdown
minutes,1,15.689562,3.596156,9.773885,21.605239,"{'expected': 32.275, 'lower_90': 24.9502743395...",NaN,NaN
volume,1,15.689562,3.596156,9.773885,21.605239,"{'fga': 14.579675197027315, 'fg3a': 5.03039607...",NaN,NaN
efficiency,1,15.689562,3.596156,9.773885,21.605239,"{'fg_pct': 0.4401805869074492, 'fg3_pct': 0.34...",NaN,NaN
matchup,1,15.689562,3.596156,9.773885,21.605239,NaN,"{'pace': 1.0, 'defense': 1.0, 'foul_rate': 1.0...",NaN
usage,1,15.689562,3.596156,9.773885,21.605239,NaN,1.0,NaN
is_b2b,1,15.689562,3.596156,9.773885,21.605239,NaN,False,NaN
blowout_prob,1,15.689562,3.596156,9.773885,21.605239,NaN,0.15,NaN
from_2pt,1,15.689562,3.596156,9.773885,21.605239,NaN,NaN,8.406815
from_3pt,1,15.689562,3.596156,9.773885,21.605239,NaN,NaN,5.193721
from_ft,1,15.689562,3.596156,9.773885,21.605239,NaN,NaN,2.089027


In [3]:
from src.points_model.utils import generate_synthetic_game_logs

# Generate synthetic data with default parameters
game_logs = generate_synthetic_game_logs()

# Generate 50 players with 30 games each
game_logs = generate_synthetic_game_logs(n_players=50, games_per_player=30)
print(f"Created {len(game_logs)} game logs for {game_logs['player_id'].nunique()} players")


Created 1500 game logs for 50 players


In [13]:
def get_game_spread(team_abbrev, current_date, team_lines_dir='data/raw/team_lines'):
    """
    Get the spread for a team's game on a given date.
    """
    # Convert date to file format (YYYYMMDD)
    date_str = current_date.replace('-', '')
    
    # Find all files matching the date pattern
    team_lines_path = Path(team_lines_dir)
    pattern = f'NBA_{date_str}_*.json'
    matching_files = list(team_lines_path.glob(pattern))
    
    if not matching_files:
        return None
    
    # Get the latest file (by modification time, or by time in filename)
    # Option 1: By modification time (most recent scrape)
    latest_file = max(matching_files, key=lambda p: p.stat().st_mtime)
    
    # Option 2: By time in filename (if you prefer)
    # latest_file = max(matching_files, key=lambda p: int(p.stem.split('_')[-1]) if p.stem.split('_')[-1].isdigit() else 0)
    
    try:
        with open(latest_file, 'r') as f:
            games_data = json.load(f)
        
        # Map team abbreviations to full names
        team_name_map = {
            'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
            'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
            'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
            'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
            'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
            'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
            'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
            'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
            'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
            'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
        }
        
        team_full_name = team_name_map.get(team_abbrev)
        if not team_full_name:
            return None
        
        # Find the game with this team
        for game in games_data:
            is_home = game['home_team'] == team_full_name
            is_away = game['away_team'] == team_full_name
            
            if is_home or is_away:
                # Get spread from first bookmaker (or average across bookmakers)
                for bookmaker in game['bookmakers']:
                    for market in bookmaker['markets']:
                        if market['market_key'] == 'spreads':
                            for outcome in market['outcomes']:
                                if outcome['name'] == team_full_name:
                                    spread = outcome['point']
                                    # Spread is from team's perspective
                                    # Negative = favored (expected to win by that amount)
                                    # Positive = underdog (expected to lose by that amount)
                                    return spread
        return None
    except Exception as e:
        return None

def calculate_blowout_prob_from_spread(spread):
    """
    Calculate blowout probability from spread.
    
    Blowout = |actual margin| > 20
    Using spread as expected margin, calculate probability of blowout.
    """
    if spread is None:
        return 0.15  # Default fallback
    
    # Expected margin from team's perspective
    # If spread is -4.5, team is expected to win by 4.5
    # If spread is +4.5, team is expected to lose by 4.5
    expected_margin = -spread  # Flip sign: negative spread = positive margin
    
    # Typical NBA game margin std dev is ~12 points
    margin_std = 12.0
    
    # Probability of blowout = P(|margin| > 20)
    # This is P(margin > 20) + P(margin < -20)
    prob_win_blowout = 1 - stats.norm.cdf(20, expected_margin, margin_std)
    prob_loss_blowout = stats.norm.cdf(-20, expected_margin, margin_std)
    blowout_prob = prob_win_blowout + prob_loss_blowout
    
    # Clamp between reasonable bounds
    return max(0.05, min(0.40, blowout_prob))

### Generate different scenrios for a certain player

In [14]:
s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv')

player_id = s26[s26['PLAYER_NAME'] == 'Anthony Davis']['PLAYER_ID'].iloc[0]
market_line = 24.5
market_juice = -110

In [8]:
# =============================================================================
# PLAYER SCENARIO ANALYSIS FOR UPCOMING GAME
# =============================================================================

from src.points_model import PointsPropModel
from src.utils.helper_functions import findOpp
from datetime import datetime


def parse_minutes(min_str):
    if pd.isna(min_str): return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player_name',
    'FGA': 'fga', 'FG3A': 'fg3a', 'FTA': 'fta',
    'FGM': 'fgm', 'FG3M': 'fg3m', 'FTM': 'ftm',
    'PTS': 'pts', 'PLUS_MINUS': 'margin'
})

# Get team mapping
team_abbrev_to_id = s26_prepped.groupby('TEAM_ABBREVIATION')['TEAM_ID'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Fit model
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

# Players info
player_name = s26_prepped[s26_prepped['player_id'] == player_id]['player_name'].iloc[0] if len(s26_prepped[s26_prepped['player_id'] == player_id]) > 0 else "LeBron James"
current_date = datetime.now().strftime('%Y-%m-%d')

# Get actual game info for today
player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
is_b2b_actual = False
if not player_games.empty:
    latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
    current_date_dt = pd.to_datetime(current_date)
    days_since_last_game = (current_date_dt - latest_game_date).days
    is_b2b_actual = (days_since_last_game == 1)

# Get opponent info
opp_abbrev, home_flag = findOpp(player_name, s26, current_date)
opp_team_id = None
if opp_abbrev and opp_abbrev in team_abbrev_to_id:
    opp_team_id = int(team_abbrev_to_id[opp_abbrev])

# Get actual opponent stats (if available)
opp_pace_actual = None
opp_drtg_actual = None
if opp_team_id:
    opp_stats = model.matchup_adjuster.get_team_stats(opp_team_id)
    if opp_stats is not None:
        opp_pace_actual = opp_stats.get('PACE', 100.0)
        opp_drtg_actual = opp_stats.get('DEF_RATING', 114.0)

# Get actual spread for blowout calculation
player_team_abbrev = name_to_team.get(player_name)
spread_actual = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
blowout_prob_actual = calculate_blowout_prob_from_spread(spread_actual)

# Define scenarios
scenarios = []

# Scenario 1: ACTUAL GAME CONDITIONS (baseline)
scenarios.append({
    'name': '🎯 ACTUAL GAME CONDITIONS',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': f'B2B: {is_b2b_actual}, Blowout: {blowout_prob_actual:.1%}, Opp: {opp_abbrev or "Unknown"}'
})

# Scenario 2: If it was a back-to-back (even if it's not)
scenarios.append({
    'name': '⚠️ IF BACK-TO-BACK',
    'is_b2b': True,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'Same game but on B2B'
})

# Scenario 3: Higher blowout risk
scenarios.append({
    'name': '📉 HIGH BLOWOUT RISK',
    'is_b2b': is_b2b_actual,
    'blowout_prob': min(0.40, blowout_prob_actual + 0.20),
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'Increased blowout probability'
})

# Scenario 4: Fast-paced opponent
scenarios.append({
    'name': '⚡ FAST-PACED OPPONENT',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': (opp_pace_actual or 100.0) + 8,  # +8 possessions
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'More possessions = more opportunities'
})

# Scenario 5: Weak defense opponent
scenarios.append({
    'name': '🛡️ WEAK DEFENSE',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': (opp_drtg_actual or 114.0) + 4,  # Higher DRTG = worse defense
    'usage_adjustment': 1.0,
    'description': 'Easier scoring opportunities'
})

# Scenario 6: Strong defense opponent
scenarios.append({
    'name': '🛡️ STRONG DEFENSE',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': (opp_drtg_actual or 114.0) - 4,  # Lower DRTG = better defense
    'usage_adjustment': 1.0,
    'description': 'Tougher matchup'
})

# Scenario 7: Star teammate out (usage boost)
scenarios.append({
    'name': '⭐ STAR TEAMMATE OUT',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.15,  # 15% usage boost
    'description': 'More touches and shots'
})

# Scenario 8: Best case scenario
scenarios.append({
    'name': '🚀 BEST CASE',
    'is_b2b': False,
    'blowout_prob': 0.05,
    'opp_pace': (opp_pace_actual or 100.0) + 8,
    'opp_drtg': (opp_drtg_actual or 114.0) + 4,
    'usage_adjustment': 1.15,
    'description': 'Rested + fast pace + weak defense + usage boost'
})

# Scenario 9: Worst case scenario
scenarios.append({
    'name': '📉 WORST CASE',
    'is_b2b': True,
    'blowout_prob': 0.35,
    'opp_pace': (opp_pace_actual or 100.0) - 5,
    'opp_drtg': (opp_drtg_actual or 114.0) - 4,
    'usage_adjustment': 0.9,
    'description': 'B2B + blowout risk + slow pace + strong defense'
})

# Generate projections for all scenarios
results = []
for scenario in scenarios:
    projection = model.project_points(
        player_id=player_id,
        is_b2b=scenario['is_b2b'],
        blowout_prob=scenario['blowout_prob'],
        opp_pace=scenario['opp_pace'],
        opp_drtg=scenario['opp_drtg'],
        usage_adjustment=scenario['usage_adjustment'],
        opp_team_id=opp_team_id if scenario['opp_pace'] == (opp_pace_actual or 100.0) else None
    )
    
    if projection:
        results.append({
            'scenario': scenario['name'],
            'expected_points': projection['expected_points'],
            'std': projection['std'],
            'lower_90': projection['lower_90'],
            'upper_90': projection['upper_90'],
            'minutes': projection['components']['minutes']['expected'],
            'fga': projection['components']['volume']['fga'],
            'fg3a': projection['components']['volume']['fg3a'],
            'fta': projection['components']['volume']['fta'],
            'description': scenario.get('description', '')
        })

# Display results
scenarios_df = pd.DataFrame(results)
scenarios_df = scenarios_df.sort_values('expected_points', ascending=False)

print("\n" + "="*100)
print(f"{s26_prepped[s26_prepped['player_id'] == player_id]['player_name'].iloc[0]} (ID: {player_id}) - SCENARIO ANALYSIS FOR UPCOMING GAME")
print(f"Date: {current_date} | Opponent: {opp_abbrev or 'Unknown'}")
print("="*100)
print(scenarios_df[['scenario', 'expected_points', 'std', 'lower_90', 'upper_90', 'minutes', 'fga', 'description']].to_string(index=False))

# Summary statistics
print(f"\n📊 PROJECTION RANGE:")
print(f"   Minimum: {scenarios_df['expected_points'].min():.1f} pts ({scenarios_df.loc[scenarios_df['expected_points'].idxmin(), 'scenario']})")
print(f"   Maximum: {scenarios_df['expected_points'].max():.1f} pts ({scenarios_df.loc[scenarios_df['expected_points'].idxmax(), 'scenario']})")
print(f"   Average: {scenarios_df['expected_points'].mean():.1f} pts")
print(f"   Range: {scenarios_df['expected_points'].max() - scenarios_df['expected_points'].min():.1f} pts")
print(f"   Actual Game: {scenarios_df[scenarios_df['scenario'].str.contains('ACTUAL')]['expected_points'].iloc[0]:.1f} pts")

print(f"\n" + "="*100)
print(f"PROP EVALUATION: Over/Under {market_line} @ {market_juice}")
print("="*100)

for scenario in scenarios:
    evaluation = model.evaluate_prop(
        player_id=player_id,
        market_line=market_line,
        market_juice=market_juice,
        is_b2b=scenario['is_b2b'],
        blowout_prob=scenario['blowout_prob'],
        opp_pace=scenario['opp_pace'],
        opp_drtg=scenario['opp_drtg'],
        usage_adjustment=scenario['usage_adjustment'],
        opp_team_id=opp_team_id if scenario['opp_pace'] == (opp_pace_actual or 100.0) else None
    )
    
    if evaluation:
        edge = evaluation['edge_analysis']
        proj = evaluation['projection']
        print(f"\n{scenario['name']}:")
        print(f"  Projection: {proj['expected_points']:.1f} pts")
        print(f"  P(Over {market_line}): {edge['prob_over']:.1%}")
        print(f"  P(Under {market_line}): {edge['prob_under']:.1%}")
        print(f"  Over Edge: {edge['over_edge']:+.1%}")
        print(f"  Under Edge: {edge['under_edge']:+.1%}")
        print(f"  Recommendation: {edge['recommendation']}")
        print(f"  EV: {edge['expected_value']:+.3f}")
        if edge.get('kelly_bet_size', 0) > 0:
            print(f"  Kelly: {edge['kelly_bet_size']*100:.1f}% ({edge.get('unit_recommendation', 'N/A')})")


Anthony Davis (ID: 203076) - SCENARIO ANALYSIS FOR UPCOMING GAME
Date: 2025-12-15 | Opponent: UTA
                scenario  expected_points      std  lower_90  upper_90   minutes       fga                                     description
             🚀 BEST CASE        26.819141 6.348714 16.375506 37.262777 31.048889 23.328027 Rested + fast pace + weak defense + usage boost
     ⭐ STAR TEAMMATE OUT        24.107147 5.985929 14.260293 33.954000 30.890577 20.750867                          More touches and shots
   ⚡ FAST-PACED OPPONENT        22.356937 5.761642 12.879036 31.834839 30.890577 19.446679           More possessions = more opportunities
         🛡️ WEAK DEFENSE        21.755178 5.686516 12.400859 31.109498 30.890577 18.726348                    Easier scoring opportunities
🎯 ACTUAL GAME CONDITIONS        20.962736 5.589247 11.768425 30.157047 30.890577 18.044232            B2B: False, Blowout: 10.3%, Opp: UTA
     📉 HIGH BLOWOUT RISK        20.555569 5.540033 11.442215 29.668

In [21]:
from src.points_model.bet_tracker import LineTracker, PropBetTracker

line_tracker = LineTracker(lines_directory='data/raw/player_lines')
player_name = 'P.J. Washington'
player_id = s26[s26['PLAYER_NAME'] == player_name]['PLAYER_ID'].iloc[0]
prop_type = 'player_points'
game_date = '2025-12-16'
file_type = 'DFS'

# First, let's debug - see what data is available
lines = line_tracker.load_all_lines(file_type=file_type)
print(f"Total lines loaded: {len(lines)}")

brunson_lines = lines[(lines['NAME'] == player_name) & (lines['CATEGORY'] == prop_type)]
if not brunson_lines.empty:
    print(f"\n{player_name} available game dates:")
    print(brunson_lines['COMMENCE_TIME'].unique())
    print(f"\nSample data:")
    print(brunson_lines[['NAME', 'COMMENCE_TIME', 'LINE', 'BOOKMAKER', 'pulled_at']].head(10))
else:
    print("No data found for Jalen Brunson")

# Use the CORRECT game date (2025-12-17, not 2025-12-14)
line_info = line_tracker.get_opening_closing_lines(
    player_name=player_name,
    prop_type=prop_type,
    game_date=game_date  # Use the actual game date from COMMENCE_TIME
)

# Or remove date filter to see all games
# line_info = line_tracker.get_opening_closing_lines(
#     player_name='Jalen Brunson',
#     prop_type='player_points'
# )

# This returns a dict with opening/closing info by bookmaker
if line_info:
    for book, info in line_info.items():
        print(f"\n{book}:")
        print(f"  Opening: {info['opening_line']} @ {info['opening_time']}")
        print(f"  Closing: {info['closing_line']} @ {info['closing_time']}")
        print(f"  Movement: {info['line_movement']:+.1f} points")
else:
    print("\nNo line info found. Check:")
    print("1. Player name spelling (exact match required)")
    print("2. Game date matches COMMENCE_TIME in data")
    print("3. File type (DFS vs US)")

Total lines loaded: 16825

P.J. Washington available game dates:
['2025-12-16']

Sample data:
                 NAME COMMENCE_TIME  LINE         BOOKMAKER  \
9033  P.J. Washington    2025-12-16  15.5  DraftKings Pick6   
9034  P.J. Washington    2025-12-16  15.5  DraftKings Pick6   
8602  P.J. Washington    2025-12-16  15.5        PrizePicks   
8603  P.J. Washington    2025-12-16  15.5        PrizePicks   
8877  P.J. Washington    2025-12-16  15.5          Betr DFS   
8876  P.J. Washington    2025-12-16  15.5          Betr DFS   
3448  P.J. Washington    2025-12-16  16.5          Betr DFS   
3447  P.J. Washington    2025-12-16  16.5          Betr DFS   
3204  P.J. Washington    2025-12-16  16.5        PrizePicks   
3203  P.J. Washington    2025-12-16  16.5        PrizePicks   

               pulled_at  
9033 2025-12-15 11:16:30  
9034 2025-12-15 11:16:30  
8602 2025-12-15 11:16:30  
8603 2025-12-15 11:16:30  
8877 2025-12-15 11:16:30  
8876 2025-12-15 11:16:30  
3448 2025-12-15 15:02:0

In [2]:
from src.points_model.bet_tracker import PropBetTracker
from src.points_model import PointsPropModel
import pandas as pd

# Load your ev_analysis file
ev_data = pd.read_csv('data/props/ev_analysis/prizepicks.csv')

# Load s26 for actual results and to get player_id
s26 = pd.read_csv('data/raw/seaon_stats/S26.csv')

# Prepare s26 data for model (same as production notebook)
def parse_minutes(min_str):
    """Parse minutes string like '35:30' to float"""
    if pd.isna(min_str):
        return 0.0
    if isinstance(min_str, (int, float)):
        return float(min_str)
    parts = str(min_str).split(':')
    if len(parts) == 2:
        return float(parts[0]) + float(parts[1]) / 60.0
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_NAME': 'player_name',
    'PLAYER_ID': 'player_id',
    'GAME_DATE': 'game_date',
    'PTS': 'points',
    'MIN': 'min',
    'FGA': 'fga',
    'FGM': 'fgm',
    'FG3A': 'fg3a',
    'FG3M': 'fg3m',
    'FTA': 'fta',
    'FTM': 'ftm',
})

# Create name/id mappings
name_to_id = s26_prepped.groupby('player_name')['player_id'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Load and fit model
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

# Initialize tracker
tracker = PropBetTracker(filepath='data/prop_bets.csv')

# Function to get projected components from model
def get_projected_components(player_id, game_date, model, s26_prepped):
    """Get projected minutes and FGA from model"""
    try:
        # Get player logs for context
        player_logs = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('game_date')
        
        # Check if back-to-back
        is_b2b = False
        if not player_logs.empty:
            latest_game_date = pd.to_datetime(player_logs['game_date'].iloc[-1])
            current_date_dt = pd.to_datetime(game_date)
            days_since_last_game = (current_date_dt - latest_game_date).days
            is_b2b = (days_since_last_game == 1)
        
        # Get projection
        proj = model.project_points(
            player_id=player_id,
            is_b2b=is_b2b
        )
        
        if proj:
            return {
                'mins_proj': proj['components']['minutes']['expected'],
                'fga_proj': proj['components']['volume']['fga']
            }
    except Exception as e:
        print(f"Error getting projection for player_id {player_id}: {e}")
    
    return {'mins_proj': None, 'fga_proj': None}

# Function to get actual results
def get_actual_result(player_name, game_date, s26_data):
    """Get actual game result from s26"""
    player_data = s26_data[
        (s26_data['PLAYER_NAME'] == player_name) & 
        (s26_data['GAME_DATE'] == game_date)
    ]
    
    if len(player_data) == 0:
        return None
    
    actual = player_data.iloc[0]
    return {
        'actual_points': actual['PTS'],
        'actual_minutes': actual['MIN'],
        'actual_fga': actual['FGA'],
        'team': actual['TEAM_ABBREVIATION'],
        'opponent': actual['OPP_ABBREVIATION'],
        'team_score': actual['TEAM_PTS'],
        'opp_score': actual['OPP_PTS'],
        'opp_pace': actual['OPP_PACE'],
        'opp_drtg': actual['OPP_DEF_RATING']
    }

# Add bets from ev_analysis
game_date = '2025-12-15'  # The game date

for _, row in ev_data.iterrows():
    player_name = row['NAME']
    
    # Get player_id
    player_id = name_to_id.get(player_name)
    if player_id is None:
        print(f"Warning: No player_id found for {player_name}")
        continue
    
    # Get projected min/fga from model
    components = get_projected_components(player_id, game_date, model, s26_prepped)
    
    # Get player data from s26 to get team/opponent/scores
    player_s26 = s26[
        (s26['PLAYER_NAME'] == player_name) & 
        (s26['GAME_DATE'] == game_date)
    ]
    
    if len(player_s26) == 0:
        print(f"Warning: No s26 data found for {player_name} on {game_date}")
        continue
    
    player_row = player_s26.iloc[0]
    
    # Get team, opponent, and scores from s26
    team = row.get('TEAM', player_row['TEAM_ABBREVIATION'])
    opponent = player_row['OPP_ABBREVIATION']
    team_score = player_row['TEAM_PTS']
    opp_score = player_row['OPP_PTS']
    
    # Create bet data
    bet_data = {
        'date': game_date,
        'player': player_name,
        'team': team,
        'opponent': opponent,
        'player_id': player_id,
        'prop_type': 'points',
        'side': row['SIDE'].upper(),
        'bookmaker': 'PrizePicks',
        'line_at_bet': row['LINE'],
        'odds_at_bet': row.get('PRIZEPICKS_ODDS', -137),
        'projection': row['PREDICTION'],  # From ev_analysis
        'projection_std': row['STD'],  # From ev_analysis
        'predicted_prob': row['MODEL_PROB'],  # From ev_analysis
        'predicted_edge': row['EDGE_VS_FAIR'],  # From ev_analysis
        'units_bet': 1.0,
        'mins_proj': components['mins_proj'],  # From model
        'fga_proj': components['fga_proj'],  # From model
        'team_score': team_score,
        'opp_score': opp_score,
    }
    
    bet_index = tracker.add_bet(bet_data)
    print(f"✓ Added bet: {player_name} {row['SIDE']} {row['LINE']} (mins: {components['mins_proj']:.1f}, fga: {components['fga_proj']:.1f})")

# After the game, update with actual results
pending_bets = tracker.get_pending_bets()

print(f"\nUpdating {len(pending_bets)} pending bets with actual results...")

for idx, bet in pending_bets.iterrows():
    actual = get_actual_result(bet['player'], str(bet['date'])[:10], s26)
    
    if actual:
        tracker.update_result(
            bet_index=idx,
            actual_result=actual['actual_points'],
            mins_actual=actual['actual_minutes'],
            fga_actual=actual['actual_fga']
        )
        print(f"✓ Updated: {bet['player']} - {actual['actual_points']} pts")
    else:
        print(f"⚠ No actual data for {bet['player']} on {bet['date']}")

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/bet_tracker.py:292: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.bets = pd.concat([self.bets, new_bet], ignore_index=True)


✓ Added bet: P.J. Washington UNDER 18.5 pts
✓ Added bet: P.J. Washington Under 18.5 (mins: 32.7, fga: 14.2)
✓ Added bet: Kyle Filipowski UNDER 12.5 pts
✓ Added bet: Kyle Filipowski Under 12.5 (mins: 22.3, fga: 7.1)
✓ Added bet: Klay Thompson UNDER 13.5 pts
✓ Added bet: Klay Thompson Under 13.5 (mins: 21.1, fga: 10.1)
✓ Added bet: Peyton Watson UNDER 12.5 pts
✓ Added bet: Peyton Watson Under 12.5 (mins: 26.3, fga: 7.8)
✓ Added bet: Cameron Johnson UNDER 14.0 pts
✓ Added bet: Cameron Johnson Under 14.0 (mins: 30.9, fga: 8.8)
✓ Added bet: Cooper Flagg UNDER 20.5 pts
✓ Added bet: Cooper Flagg Under 20.5 (mins: 33.2, fga: 14.6)
✓ Added bet: Naji Marshall UNDER 15.5 pts
✓ Added bet: Naji Marshall Under 15.5 (mins: 26.9, fga: 8.1)
✓ Added bet: Ace Bailey UNDER 12.5 pts
✓ Added bet: Ace Bailey Under 12.5 (mins: 23.3, fga: 8.7)
✓ Added bet: Max Christie OVER 11.5 pts
✓ Added bet: Max Christie Over 11.5 (mins: 28.7, fga: 8.0)
✓ Added bet: Ja Morant UNDER 19.5 pts
✓ Added bet: Ja Morant Under 19.

In [ ]:
from src.points_model.bet_tracker import PerformanceAnalyzer

# Initialize analyzer
analyzer = PerformanceAnalyzer(tracker)

# Run full analysis
analyzer.full_report()

# Or run individual analyses:
analyzer.summary(last_n_days=30)  # Last 30 days
analyzer.calibration_analysis()  # Check if probabilities are accurate
analyzer.edge_bucket_analysis()  # Performance by edge size
analyzer.clv_analysis()  # Closing Line Value analysis
analyzer.component_attribution()  # Where errors come from